In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Veri Yükleme ve Temizlik
df = pd.read_csv("Salary_Data.csv")
df = df.dropna(subset=["Salary"]).reset_index(drop=True)

edu_map = {
    "High School": 0,
    "Bachelor's": 1,
    "Bachelor's Degree": 1,
    "Master's": 2,
    "Master's Degree": 2,
    "PhD": 3,
    "phd": 3
}
df["Education_Clean"] = df["Education Level"].map(edu_map)
df["Education_Clean"] = df["Education_Clean"].fillna(1)

features = ["Age", "Years of Experience", "Education_Clean", "Job Title"]
X = df[features]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Years of Experience", "Education_Clean"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Job Title"])
    ]
)

models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42)
}

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    
    print(f"--- {name} ---")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE: {mae:.2f}\n")
